# IT2226 — Progress Review 1 : classification d'images dentaires

**Auteur :** _ton nom_  |  **Modèle :** ResNet-50 (changer `MODEL_NAME` pour EfficientNet-B0)  |  **Dataset :** Kaggle Oral Image Dataset (6 classes)

## Ce que ce notebook démontre (grille du PR1)
| Section | Critère noté |
|---|---|
| 1-2. Données et EDA | Collecte, EDA avec visualisations |
| 3. Préparation | Split sans fuite de données, préparation propre |
| 4-6. Entraînement | Entraînement réel (pas juste un modèle pré-entraîné), preuves d'optimisation |
| 7. Évaluation | Métriques adaptées et bien interprétées, sur le **test** uniquement |
| Tout | Structure, commentaires, markdown |

## Comment l'utiliser
1. Colab : **Exécution > Modifier le type d'exécution > GPU (T4)**.
2. Exécute les cellules **dans l'ordre**, en lisant le markdown avant chaque cellule.
3. Aux endroits marqués ✍️, **écris toi-même** l'interprétation : c'est ce qui est noté, et c'est ce que tu devras défendre à l'oral.

> ⚠️ Ce code a été généré avec l'aide de Claude. Au PR1 tu l'expliqueras **sans notes ni IA**. Avant de déposer, tu dois pouvoir dire pour chaque cellule : ce qu'elle fait, pourquoi elle est là, et ce qui casse si on l'enlève. Note tes vérifications dans ton AI Use Log.

## 1. Mise en place

On installe `kagglehub` (téléchargement du dataset), on fixe les paramètres **en un seul endroit** et on vérifie le GPU.

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, random, hashlib, copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ---------- Paramètres (tout est ici : c'est le premier endroit à modifier lors d'une tâche en direct) ----------
SEED = 42                    # graine : rend le split et l'entraînement reproductibles
IMG_SIZE = 224               # taille d'entrée attendue par ResNet-50 / EfficientNet pré-entraînés
BATCH_SIZE = 32              # baisser à 16 si "CUDA out of memory"
NUM_WORKERS = 2              # processus de chargement des images (mettre 0 en cas de souci)
MODEL_NAME = "resnet50"      # "resnet50" ou "efficientnet_b0"
TEST_FRACTION = 0.15         # 15 % test
VAL_FRACTION = 0.15          # 15 % validation (le reste, 70 %, sert à l'entraînement)

OUT_DIR = Path("outputs")    # figures et modèle sauvegardés ici (à télécharger avant de fermer Colab !)
OUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", DEVICE, "| torch", torch.__version__)
if DEVICE.type == "cpu":
    print("ATTENTION : pas de GPU. Active-le dans Exécution > Modifier le type d'exécution.")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## 2. Les données

### 2.1 Téléchargement
Le dataset est public (licence CC0). On le télécharge depuis Kaggle directement dans Colab : **on ne le met jamais dans GitHub** (473 Mo).

Si le téléchargement demande une authentification : sur Kaggle, *Settings > API > Create New Token*, puis dans Colab (icône 🔑 des *Secrets*) ajoute `KAGGLE_USERNAME` et `KAGGLE_KEY`.

In [ ]:
import kagglehub

try:
    DATA_ROOT = Path(kagglehub.dataset_download("jhirshaikh/oral-image-dataset"))
except Exception as e:
    print("Téléchargement impossible :", e)
    print("-> Mets le dataset dans Colab à la main et indique son dossier ci-dessous.")
    DATA_ROOT = Path("/content/oral-image-dataset")   # <-- à adapter si besoin

print("Dataset dans :", DATA_ROOT)

### 2.2 Inventaire des images
On parcourt tous les dossiers et on construit un tableau `df` : **une ligne = une image**, avec son chemin et sa classe.
La classe est le **nom du dossier qui contient l'image**. Si le dataset est déjà découpé en `train/` et `test/`, on les fusionne : on refera notre propre split à l'étape 3, contrôlé par nous.

**À vérifier à l'œil dans la sortie :** on doit trouver les 6 classes attendues (Calculus, Caries, Hypodontia, Mouth Ulcer, Normal, Tooth Discoloration) et environ 4 375 images.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_FOLDERS = {"train", "test", "val", "valid", "validation"}   # dossiers de découpage, pas des classes

rows = []
for p in DATA_ROOT.rglob("*"):
    if p.suffix.lower() in IMG_EXT:
        label = p.parent.name
        if label.lower() in SPLIT_FOLDERS:      # image sans dossier de classe : on l'ignore
            continue
        rows.append({"path": str(p), "label": label})

df = pd.DataFrame(rows)
assert len(df) > 0, "Aucune image trouvée : vérifie DATA_ROOT"
print(f"{len(df)} images, {df.label.nunique()} classes")
print(df.label.value_counts())
if df.label.nunique() != 6:
    print("ATTENTION : on attendait 6 classes. Regarde les noms ci-dessus et la structure des dossiers.")

## 3. EDA — Exploration des données

But : **comprendre les données avant de modéliser**. Chaque graphique doit mener à une observation écrite par toi (✍️ en bas de section).

### 3.1 Combien d'images par classe ?
Le brief annonce un déséquilibre (Calculus ≈ 1 296 images, Caries ≈ 219). On le mesure.

In [ ]:
counts = df.label.value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(counts.index, counts.values, color="#4C78A8")
ax.bar_label(bars)
ax.set_title("Nombre d'images par classe")
ax.set_ylabel("images")
plt.xticks(rotation=30, ha="right")
fig.tight_layout()
fig.savefig(OUT_DIR / "eda_class_counts.png", dpi=150)
plt.show()

print(f"Rapport plus grande / plus petite classe : {counts.max() / counts.min():.1f}x")

### 3.2 À quoi ressemblent les images ?
5 images au hasard par classe. On regarde : luminosité, cadrage, présence de plusieurs conditions sur une même photo, images qui semblent mal étiquetées.

In [ ]:
classes_sorted = sorted(df.label.unique())
n_show = 5
fig, axes = plt.subplots(len(classes_sorted), n_show, figsize=(2.4 * n_show, 2.4 * len(classes_sorted)))
rng = np.random.RandomState(SEED)
for r, cls in enumerate(classes_sorted):
    paths = df[df.label == cls].path.tolist()
    for c, p in enumerate(rng.choice(paths, size=min(n_show, len(paths)), replace=False)):
        axes[r, c].imshow(Image.open(p).convert("RGB"))
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(cls, loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "eda_sample_grid.png", dpi=120)
plt.show()

### 3.3 Tailles et luminosité des images
Si les images ont des tailles très différentes, le redimensionnement à 224×224 déforme plus ou moins selon les cas. Si une classe est systématiquement plus sombre, le modèle risque d'apprendre la **luminosité** au lieu de la maladie (un biais).
On échantillonne 60 images par classe pour que ce soit rapide.

In [ ]:
stats = []
for cls in classes_sorted:
    sample = df[df.label == cls].sample(n=min(60, (df.label == cls).sum()), random_state=SEED)
    for p in sample.path:
        img = Image.open(p)
        gray = np.asarray(img.convert("L"), dtype=np.float32)
        stats.append({"label": cls, "width": img.width, "height": img.height, "brightness": gray.mean()})
stats = pd.DataFrame(stats)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
stats.boxplot(column="brightness", by="label", ax=ax[0], rot=30)
ax[0].set_title("Luminosité moyenne par classe"); ax[0].set_xlabel(""); fig.suptitle("")
ax[1].scatter(stats.width, stats.height, s=8, alpha=0.5)
ax[1].set_title("Largeur x hauteur des images"); ax[1].set_xlabel("largeur"); ax[1].set_ylabel("hauteur")
fig.tight_layout()
fig.savefig(OUT_DIR / "eda_size_brightness.png", dpi=150)
plt.show()

print(stats.groupby("label")[["width", "height", "brightness"]].mean().round(1))

### 3.4 Doublons et fuite de données
**Pourquoi c'est critique :** si une même image est dans le train **et** dans le test, le modèle la « connaît » déjà et le score de test est faussé (fuite de données). Le guidebook cite cette erreur comme l'une des pires.

- **Doublons exacts** : même contenu octet par octet (empreinte MD5). On les **supprime** avant de découper.
- **Quasi-doublons** : images presque identiques (empreinte visuelle 8×8). On les **compte seulement** : cette méthode simple peut avoir des fausses alertes, donc on ne supprime pas à l'aveugle.
- **Conflits d'étiquettes** : la même image classée dans deux classes différentes.

Cette cellule lit tous les fichiers : compte 1 à 2 minutes.

In [ ]:
def md5_of(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def ahash_of(path):
    # empreinte visuelle : image réduite à 8x8 en gris, chaque pixel comparé à la moyenne -> 64 bits
    small = np.asarray(Image.open(path).convert("L").resize((8, 8)), dtype=np.float32)
    return "".join("1" if v else "0" for v in (small > small.mean()).flatten())


df["md5"] = df.path.map(md5_of)
df["ahash"] = df.path.map(ahash_of)

n_exact = df.duplicated("md5", keep="first").sum()
n_near = df.duplicated("ahash", keep="first").sum()
conflicts = df.groupby("md5").label.nunique()
n_conflict = int((conflicts > 1).sum())

print(f"Doublons exacts (à supprimer)        : {n_exact}")
print(f"Quasi-doublons (à titre informatif)  : {n_near}")
print(f"Images identiques avec étiquettes différentes : {n_conflict}")

✍️ **Observations de l'EDA (à écrire toi-même, 3 à 5 phrases)** — double-clique sur cette cellule et réponds :
- Quelle est l'ampleur du déséquilibre et quelle conséquence sur l'entraînement ?
- Les images sont-elles homogènes (taille, luminosité) ? Une classe se distingue-t-elle par autre chose que la maladie ?
- Combien de doublons as-tu trouvés et que fais-tu ?
- Qu'est-ce qui te surprend dans les images ?

_(Ton texte ici.)_

## 4. Préparation des données

### 4.1 Suppression des doublons exacts et encodage des classes
Le modèle manipule des nombres : chaque classe reçoit un entier (0 à 5). `sorted` rend l'ordre reproductible.

In [ ]:
before = len(df)
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)
print(f"{before - len(df)} doublons exacts supprimés -> {len(df)} images")

classes = sorted(df.label.unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["y"] = df.label.map(class_to_idx)
print(class_to_idx)

### 4.2 Découpage train / validation / test (stratifié)
| Jeu | Rôle | Part |
|---|---|---|
| **Train** | Le modèle apprend dessus | 70 % |
| **Validation** | On s'en sert pour **choisir** (nombre d'epochs, learning rate, meilleur modèle) | 15 % |
| **Test** | Note finale, **utilisé une seule fois à la fin** | 15 % |

**Stratifié** = chaque jeu garde les mêmes proportions de classes. C'est indispensable ici : sans cela, la petite classe Caries pourrait presque disparaître du test.

In [ ]:
temp_fraction = VAL_FRACTION + TEST_FRACTION
train_df, temp_df = train_test_split(df, test_size=temp_fraction, stratify=df.y, random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=TEST_FRACTION / temp_fraction, stratify=temp_df.y, random_state=SEED
)

# Contrôles : aucun chemin ni aucune empreinte commune entre les jeux
assert not set(train_df.path) & set(val_df.path) and not set(train_df.path) & set(test_df.path)
assert not set(train_df.md5) & set(val_df.md5) and not set(train_df.md5) & set(test_df.md5)
print("Aucune fuite exacte entre train, val et test.")

split_table = pd.DataFrame({
    "train": train_df.label.value_counts(),
    "val": val_df.label.value_counts(),
    "test": test_df.label.value_counts(),
}).fillna(0).astype(int)
split_table.loc["TOTAL"] = split_table.sum()
split_table

### 4.3 Transformations et chargeurs de données
- **Train** : on applique de l'**augmentation** (recadrage aléatoire, miroir, légère rotation, variations de luminosité). Cela multiplie les variantes vues par le modèle et réduit le surapprentissage.
- **Val et test** : **aucune** augmentation, seulement redimensionnement : on veut mesurer le modèle sur des images « normales » et de façon reproductible.
- **Normalisation** avec la moyenne et l'écart-type d'ImageNet : les modèles pré-entraînés ont été entraînés ainsi, il faut respecter leur format d'entrée.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # statistiques ImageNet

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class OralDataset(Dataset):
    # Un Dataset PyTorch = "donne-moi l'image n et son étiquette"
    def __init__(self, frame, transform):
        self.paths = frame.path.tolist()
        self.labels = frame.y.tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]


pin = DEVICE.type == "cuda"
train_loader = DataLoader(OralDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(OralDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(OralDataset(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=pin)
print(len(train_df), "images train |", len(val_df), "val |", len(test_df), "test")

### 4.4 Poids de classes
Sans correction, le modèle peut ignorer la petite classe (Caries) et quand même avoir une bonne accuracy. On donne donc **plus de poids aux erreurs sur les classes rares** dans la fonction de perte.
Formule : `poids = nb_total / (nb_classes × nb_images_de_la_classe)`. Calculé **uniquement sur le train** (calculer sur tout le dataset serait une petite fuite).

In [ ]:
train_counts = np.bincount(train_df.y, minlength=len(classes))
class_weights = torch.tensor(train_counts.sum() / (len(classes) * train_counts), dtype=torch.float32)
print(pd.DataFrame({"classe": classes, "images train": train_counts, "poids": class_weights.numpy().round(2)}).to_string(index=False))

## 5. Le modèle : transfert d'apprentissage (fine-tuning)

On ne part **pas de zéro** : on prend un réseau déjà entraîné sur ImageNet (14 millions d'images). Il sait déjà reconnaître des bords, des textures, des formes. On remplace seulement sa **dernière couche** par une couche à 6 sorties (nos 6 classes), puis on l'entraîne sur nos images.

**Deux phases :**
1. **Phase 1 (baseline)** : le corps du réseau (*backbone*) est **gelé**, on n'entraîne que la nouvelle couche. Rapide et stable.
2. **Phase 2 (fine-tuning)** : on **dégèle tout** avec un learning rate plus faible pour le corps (on ajuste doucement ce qu'il sait déjà, sans le casser).

*Détail à connaître pour l'oral :* en phase 1 les poids du corps sont gelés, mais ses couches de normalisation (BatchNorm) continuent de mettre à jour leurs statistiques tant que le modèle est en mode `train`. Ce n'est pas une erreur, mais c'est une différence entre « gelé » et « complètement figé ».

In [ ]:
def build_model(name, n_classes):
    # Charge un modèle pré-entraîné ImageNet et remplace sa dernière couche par n_classes sorties.
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, n_classes))
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, n_classes))
    else:
        raise ValueError("MODEL_NAME inconnu : " + name)
    return m.to(DEVICE)


def head_params(m):
    # la "tête" = la dernière couche qu'on a remplacée
    return (m.fc if hasattr(m, "fc") else m.classifier).parameters()


def set_backbone_trainable(m, trainable):
    # trainable=False -> le corps est gelé (ses poids ne changent plus), la tête reste entraînable
    head_ids = {id(p) for p in head_params(m)}
    for p in m.parameters():
        p.requires_grad = True if id(p) in head_ids else trainable


model_preview = build_model(MODEL_NAME, len(classes))
n_total = sum(p.numel() for p in model_preview.parameters())
n_head = sum(p.numel() for p in head_params(model_preview))
print(f"{MODEL_NAME} : {n_total/1e6:.1f} M de paramètres dont {n_head:,} dans la tête (phase 1 n'entraîne que ceux-là)")
del model_preview

## 6. Entraînement et optimisation

### 6.1 Fonctions d'entraînement
- `run_epoch` : une passe complète sur un jeu. Avec un `optimizer` elle **entraîne** (calcule l'erreur, rétropropage, met à jour les poids). Sans optimizer elle **évalue** seulement.
- **Métrique de sélection : macro-F1** (moyenne du F1 de chaque classe, toutes classes pesées pareil). L'accuracy seule serait trompeuse avec des classes déséquilibrées.
- `autocast` + `GradScaler` : calcul en précision mixte, plus rapide sur GPU, sans changer le résultat attendu.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, preds, labels = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                out = model(x)
                loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * x.size(0)
        preds += out.argmax(1).cpu().tolist()
        labels += y.cpu().tolist()
    n = len(labels)
    acc = float(np.mean(np.array(preds) == np.array(labels)))
    macro_f1 = f1_score(labels, preds, average="macro")
    return total_loss / n, acc, macro_f1


def run_experiment(tag, lr_head, lr_backbone, epochs_head, epochs_ft, verbose=True):
    # Un essai complet : phase 1 (tête seule) puis phase 2 (réseau entier). Retourne les résultats.
    set_seed(SEED)
    model = build_model(MODEL_NAME, len(classes))
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    history, best_f1, best_state, baseline_f1 = [], -1.0, None, None

    def one_phase(phase, epochs, optimizer, scheduler):
        nonlocal best_f1, best_state
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, optimizer, scaler)
            va_loss, va_acc, va_f1 = run_epoch(model, val_loader, criterion)
            if scheduler:
                scheduler.step()
            history.append(dict(tag=tag, phase=phase, epoch=ep, train_loss=tr_loss, val_loss=va_loss,
                                train_f1=tr_f1, val_f1=va_f1, val_acc=va_acc))
            if verbose:
                print(f"[{tag}|{phase}] epoch {ep}/{epochs} train_loss {tr_loss:.3f} | "
                      f"val_loss {va_loss:.3f} val_acc {va_acc:.3f} val_macroF1 {va_f1:.3f}")
            if va_f1 > best_f1:                       # on garde le meilleur état selon la VALIDATION
                best_f1 = va_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Phase 1 : tête seule
    set_backbone_trainable(model, False)
    opt1 = torch.optim.AdamW(head_params(model), lr=lr_head, weight_decay=1e-4)
    one_phase("phase1_tete", epochs_head, opt1, None)
    baseline_f1 = best_f1

    # Phase 2 : tout le réseau, learning rate plus faible pour le corps
    set_backbone_trainable(model, True)
    head_ids = {id(p) for p in head_params(model)}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    opt2 = torch.optim.AdamW([{"params": backbone, "lr": lr_backbone},
                              {"params": list(head_params(model)), "lr": lr_head / 10}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs_ft)
    one_phase("phase2_finetuning", epochs_ft, opt2, sched)

    return dict(tag=tag, lr_head=lr_head, lr_backbone=lr_backbone, baseline_val_f1=baseline_f1,
                best_val_f1=best_f1, best_state=best_state, history=pd.DataFrame(history))

### 6.2 Essai de test rapide (1 epoch par phase)
Avant de lancer les vrais essais (plusieurs minutes), on vérifie que tout tourne. Si ça plante ici, c'est ici qu'il faut corriger, pas après 15 minutes.

In [ ]:
t0 = time.time()
smoke = run_experiment("test_rapide", lr_head=1e-3, lr_backbone=1e-4, epochs_head=1, epochs_ft=1)
print(f"OK en {time.time() - t0:.0f} s | macro-F1 val : {smoke['best_val_f1']:.3f}")
del smoke

### 6.3 Comparaison d'hyperparamètres (la preuve d'optimisation)
Le **learning rate** (taille des pas de l'apprentissage) est l'hyperparamètre le plus influent. On teste 3 valeurs pour le corps du réseau, avec le même protocole, et on compare le **macro-F1 sur la validation**.

**Règle d'or : on choisit sur la validation, jamais sur le test.** Le test ne sert qu'à la note finale.

Durée estimée sur un GPU T4 : environ 10 à 20 minutes. Pour aller plus vite, baisse `EPOCHS_FT`.

In [ ]:
EPOCHS_HEAD = 3     # phase 1
EPOCHS_FT = 8       # phase 2
LR_HEAD = 1e-3
LR_BACKBONE_CANDIDATES = [3e-5, 1e-4, 3e-4]

experiments = []
for lr_bb in LR_BACKBONE_CANDIDATES:
    print(f"\n===== lr_backbone = {lr_bb} =====")
    experiments.append(run_experiment(f"lr_bb={lr_bb:g}", LR_HEAD, lr_bb, EPOCHS_HEAD, EPOCHS_FT))

results = pd.DataFrame([{k: v for k, v in e.items() if k not in ("best_state", "history")} for e in experiments])
results = results.sort_values("best_val_f1", ascending=False).reset_index(drop=True)
results.to_csv(OUT_DIR / "hyperparameter_results.csv", index=False)
results

In [ ]:
best = max(experiments, key=lambda e: e["best_val_f1"])
print(f"Meilleur essai (sur la VALIDATION) : {best['tag']} | macro-F1 val {best['best_val_f1']:.3f} "
      f"(baseline tête seule : {best['baseline_val_f1']:.3f})")

h = best["history"].reset_index(drop=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.train_loss, label="train"); ax[0].plot(h.val_loss, label="validation")
ax[0].axvline(EPOCHS_HEAD - 0.5, color="gray", ls="--"); ax[0].set_title("Perte (loss)"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot(h.val_f1, color="green"); ax[1].axvline(EPOCHS_HEAD - 0.5, color="gray", ls="--")
ax[1].set_title("Macro-F1 de validation"); ax[1].set_xlabel("epoch")
fig.suptitle("Pointillés : passage de la phase 1 (tête seule) à la phase 2 (fine-tuning)")
fig.tight_layout()
fig.savefig(OUT_DIR / "training_curves.png", dpi=150)
plt.show()

✍️ **Interprétation de l'entraînement (à écrire toi-même)** :
- Quel learning rate est le meilleur et de combien ? L'écart est-il grand ou dans le bruit ?
- Que gagne le fine-tuning par rapport à la baseline (tête seule) ?
- Les courbes montrent-elles du **surapprentissage** (la perte de train baisse, celle de validation remonte) ? À quelle epoch ?
- Que ferais-tu avec plus de temps ? (autre architecture, plus d'augmentation, plus d'epochs…)

_(Ton texte ici.)_

## 7. Évaluation finale sur le jeu de test

On charge le meilleur modèle choisi sur la validation et on l'évalue **une seule fois** sur le test : c'est l'estimation honnête de ses performances sur des images jamais vues.

### Comment lire les métriques
- **Précision (precision)** d'une classe : parmi les images prédites « Caries », quelle part est réellement Caries ? (Elle baisse quand le modèle voit des Caries partout.)
- **Rappel (recall)** : parmi les vraies Caries, quelle part est trouvée ? (Il baisse quand le modèle **rate** des cas, ce qui est l'erreur la plus grave en dépistage.)
- **F1** : compromis des deux. **Macro-F1** : moyenne des F1 par classe, toutes classes comptant pareil.
- **Accuracy** : part globale de bonnes réponses, trompeuse quand les classes sont déséquilibrées.
- **Matrice de confusion** : ligne = vraie classe, colonne = classe prédite. Les valeurs hors diagonale montrent quelles classes se confondent.

In [ ]:
final_model = build_model(MODEL_NAME, len(classes))
final_model.load_state_dict(best["best_state"])
final_model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = final_model(x.to(DEVICE))
        y_pred += out.argmax(1).cpu().tolist()
        y_true += y.tolist()
y_true, y_pred = np.array(y_true), np.array(y_pred)

test_acc = (y_true == y_pred).mean()
test_macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"TEST | accuracy {test_acc:.3f} | macro-F1 {test_macro_f1:.3f}\n")
report = classification_report(y_true, y_pred, target_names=classes, digits=3)
print(report)
(OUT_DIR / "classification_report.txt").write_text(report)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel("Classe prédite"); ax.set_ylabel("Vraie classe"); ax.set_title("Matrice de confusion (test)")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im)
fig.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

### 7.1 Analyse d'erreurs
On regarde 8 images mal classées : ce sont elles qui expliquent les limites du modèle (photo floue, deux conditions visibles, étiquette douteuse…).

In [ ]:
wrong = np.where(y_true != y_pred)[0]
print(f"{len(wrong)} erreurs sur {len(y_true)} images de test")
show = wrong[:8]
if len(show):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6.5))
    for ax, idx in zip(axes.flatten(), show):
        ax.imshow(Image.open(test_df.path.iloc[idx]).convert("RGB"))
        ax.set_title(f"vrai : {classes[y_true[idx]]}\nprédit : {classes[y_pred[idx]]}", fontsize=9)
        ax.axis("off")
    for ax in axes.flatten()[len(show):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "error_examples.png", dpi=130)
    plt.show()

✍️ **Interprétation des résultats (à écrire toi-même — c'est le cœur de la note « Évaluation »)** :
- Quel est le macro-F1 de test et quelle est sa différence avec le macro-F1 de validation ? Un écart important serait un signal d'alerte.
- Quelle classe a le **rappel** le plus faible ? Pourquoi est-ce grave pour un outil de dépistage ?
- Quelles paires de classes se confondent dans la matrice et pourquoi (hypothèse) ?
- Que montrent les images mal classées ?
- **Limites à écrire honnêtement** : images non singapouriennes, pas de classe « gencives », une seule étiquette par image, déséquilibre, qualité de photo variable. Ces résultats ne prouvent **pas** que le modèle marche sur de vrais patients.

_(Ton texte ici.)_

## 8. Sauvegarde

On enregistre le modèle et la liste des classes pour pouvoir le réutiliser plus tard (application et chatbot du PR2).
**Le disque de Colab est effacé à la fin de la session** : télécharge le dossier `outputs/` ou copie-le sur Drive avant de fermer.

In [ ]:
import json
torch.save(best["best_state"], OUT_DIR / f"{MODEL_NAME}_dental.pt")
(OUT_DIR / "classes.json").write_text(json.dumps(classes))
pd.DataFrame(best["history"]).to_csv(OUT_DIR / "training_history.csv", index=False)
print("Fichiers sauvegardés :", sorted(p.name for p in OUT_DIR.iterdir()))

# Option : copier sur Google Drive (décommente)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs "/content/drive/MyDrive/IT2226_outputs"

## 9. Checklist avant dépôt
- [ ] Notebook exécuté **de haut en bas** sans erreur, toutes les sorties visibles
- [ ] Les 4 cellules ✍️ sont remplies **avec tes mots**
- [ ] Mon AI Use Log est à jour et cohérent avec mes commits
- [ ] Je peux expliquer chaque cellule sans notes
- [ ] `outputs/` sauvegardé hors de Colab